# LlamaParse 활용하여 PDF를 Markdown으로 추출

In [1]:
## 라이브러리 설치
# !pip install llama-parse
# !pip install langchain-text-splitters

In [2]:
## LlamaParse를 활용한 PDF 약관 파싱
import nest_asyncio
nest_asyncio.apply()

from llama_parse import LlamaParse

# 1. 파서 세팅 (마크다운 출력 및 한국어 최적화)
parser = LlamaParse(
    api_key="llx-nMZlBWxeTV11P8Zd3EIZ1vfYS1OQNfQnv1CtM36pfxfaXhBK", 
    result_type="markdown",   # 핵심: 표와 구조를 마크다운으로 유지
    language="ko",            # 한국어 인식률 향상
    verbose=True
)

# 2. 다운로드한 약관 PDF 파일 로드 및 파싱
file_path = "./자동차보험_약관.pdf"  # 실제 파일 경로로 변경
print("파싱을 시작합니다. (시간이 조금 걸릴 수 있습니다...)")
parsed_docs = parser.load_data(file_path)

# 3. 추출된 전체 마크다운 텍스트 확인
markdown_text = parsed_docs[0].text
print("--- [파싱 결과 미리보기] ---")
print(markdown_text[:1000]) # 앞부분 1,000자만 출력해보기

C:\Users\cloud\AppData\Local\Temp\ipykernel_10436\1353004786.py:5: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


파싱을 시작합니다. (시간이 조금 걸릴 수 있습니다...)
Started parsing the file under job_id 9ab74322-2f20-48b0-8fd3-b8b057d6c4fb
--- [파싱 결과 미리보기] ---
# 개인용 자동차보험 약관

# 2026.04.11 개정

이 약관은 금융소비자보호법 및 소비자보호 내부통제기준에 따른 절차를 거쳐 제공됩니다.

# 보험약관



In [3]:
# 1. 파싱된 데이터가 총 몇 장(페이지)인지 확인
print(f"✅ 총 파싱된 페이지 수: {len(parsed_docs)}장")

# 2. 모든 페이지의 텍스트를 하나의 거대한 텍스트로 합치기 (.join 활용)
full_markdown_text = "\n\n".join([doc.text for doc in parsed_docs])

# 3. 전체 글자 수 확인 및 합쳐진 본문 미리보기
print(f"✅ 전체 글자 수: {len(full_markdown_text):,}자\n")
print("--- [합쳐진 전체 마크다운 중 본문 일부 미리보기] ---")
# 표지(앞부분) 말고 진짜 본문이 나올 법한 중간 부분(예: 3000자~4000자 구간)을 잘라서 확인
print(full_markdown_text[3000:4000])

✅ 총 파싱된 페이지 수: 282장
✅ 전체 글자 수: 408,211자

--- [합쳐진 전체 마크다운 중 본문 일부 미리보기] ---
                             | p.175            |      |
| 대인배상Ⅰ                                         | 제5조              | p.57 |
| 대인Ⅱ                                           | 2 보상하지 않는 손해     |      |
| ·대물배상                                         | 제8조              | p.58 |
| 자기신체사고                                        | 제14조             | p.61 |
| \* 본인이 가입한 특약을 확인하여 가입특약별 보상하는 손해도 반드시 확인할 필요 |                  |      |
| 무보험자동차에 의한 상해                                 | 제19조             | p.63 |
| 자기차량손해                                        |                  | p.65 |
| (차대차 및 도난 사고에 한정)                             | 제23조             |      |
| ※ 차량단독사고 손해보상 특별약관                            |                  |      |
|                                               | p.176            |      |
| 3 청약 철회                                       | 제41조(청약 철회)      | p.78 |
| 4 계약 전 알릴 의무                 

In [4]:
## 마크다운 구조 살려서 쪼개기(Chunking)

from langchain_text_splitters import MarkdownHeaderTextSplitter

# 1. 쪼갤 기준이 되는 헤더 정의
headers_to_split_on = [
    ("#", "대분류"),       # 예: # 무배당 실손의료비 약관
    ("##", "조항"),        # 예: ## 제3조 (보상하는 손해)
    ("###", "세부항목"),     # 예: ### 1. 질병 통원
]

# 2. 스플리터 초기화 및 실행
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False # 헤더 텍스트를 청크 내용에서 날리지 않고 유지
)

# 3. 텍스트 분할
chunks = markdown_splitter.split_text(full_markdown_text)

# 4. 첫 번째 청크 구조 확인
print("\n--- [첫 번째 Chunk 데이터 구조] ---")
print("메타데이터:", chunks[0].metadata)
print("본문 내용:\n", chunks[0].page_content)

c:\Users\cloud\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED




--- [첫 번째 Chunk 데이터 구조] ---
메타데이터: {'대분류': '개인용 자동차보험 약관'}
본문 내용:
 # 개인용 자동차보험 약관


In [5]:
len(chunks)

1344

# 벡터로 변환

In [ ]:
# 라이브러리 설치
# !pip install langchain-chroma langchain-huggingface sentence-transformers

  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   - -------------------------------------- 1.0/23.5 MB 6.8 MB/s eta 0:00:04
   -- ------------------------------------- 1.6/23.5 MB 4.4 MB/s eta 0:00:06
   ---- ----------------------------------- 2.6/23.5 MB 4.4 MB/s eta 0:00:05
   ----- ---------------------------------- 3.4/23.5 MB 4.3 MB/s eta 0:00:05
   ------- -------------------------------- 4.5/23.5 MB 4.6 MB/s eta 0:00:05
   -------- ------------------------------- 5.2/23.5 MB 4.4 MB/s eta 0:00:05
   ----------- ---------------------------- 6.6/23.5 MB 4.5 MB/s eta 0:00:04
   ----------- ---------------------------- 6.6/23.5 MB 4.5 MB/s eta 0:00:04
   -------------- ------------------------- 8.4/23.5 MB 4.5 MB/s eta 0:00:04
   --------------- ------------------------ 9.2/23.5 MB 4.5 MB/s eta 0:00:04
   ---------------

In [ ]:
# !pip install --upgrade langsmith langchain-core langchain


   ---------------------------------------- 0/6 [ormsgpack]
   ---------------------------------------- 0/6 [ormsgpack]
   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------------- -------------------------- 2/6 [langgraph-checkpoint]
   ------------- -------------------------- 2/6 [langgraph-checkpoint]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   --

In [6]:
## Vector DB 구축

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("1. BGE-m3 임베딩 모델을 불러옵니다. (최초 실행 시 다운로드로 인해 시간이 걸립니다...)")
# 한국어와 긴 문장에 강력한 BGE-m3 모델 로드
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

print("2. 조각(Chunks)들을 벡터로 변환하여 Chroma DB에 저장합니다.")
# 앞서 만든 chunks 리스트를 DB에 통째로 밀어넣기
# persist_directory를 지정하면 데이터가 컴퓨터(로컬) 폴더에 저장되어 나중에 재사용할 수 있습니다.
vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings,
    persist_directory="./insurance_chroma_db" 
)

print("🎉 Vector DB 구축 완료! 이제 검색을 시작할 수 있습니다.")

1. BGE-m3 임베딩 모델을 불러옵니다. (최초 실행 시 다운로드로 인해 시간이 걸립니다...)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\cloud\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\cloud\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

2. 조각(Chunks)들을 벡터로 변환하여 Chroma DB에 저장합니다.
🎉 Vector DB 구축 완료! 이제 검색을 시작할 수 있습니다.


In [8]:
## 저장 
import pickle

# chunks 리스트를 'insurance_chunks.pkl' 이라는 파일로 영구 저장
with open("insurance_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("✅ 청크 백업 완료! 이제 70분 다시 안 기다려도 됩니다!")

✅ 청크 백업 완료! 이제 70분 다시 안 기다려도 됩니다!


In [9]:
# 결과를 눈으로 확인하기 위한 마크다운 파일로 저장
with open("parsed_result_readable.md", "w", encoding="utf-8") as f:
    for i, chunk in enumerate(chunks):
        f.write(f"### ✂️ [조각 번호: {i+1}]\n")
        f.write(f"**🏷️ 메타데이터 (출처/제목):** {chunk.metadata}\n\n")
        f.write(f"**📄 본문 내용:**\n{chunk.page_content}\n")
        f.write("\n" + "="*50 + "\n\n")

print("✅ 'parsed_result_readable.md' 파일이 생성되었습니다. 메모장으로 열어보세요!")

import json

# 청크 데이터를 JSON 형태로 변환
chunk_list = []
for i, chunk in enumerate(chunks):
    chunk_list.append({
        "chunk_id": i + 1,
        "metadata": chunk.metadata,
        "content": chunk.page_content
    })

# json 파일로 예쁘게(indent=4) 저장
with open("parsed_result_data.json", "w", encoding="utf-8") as f:
    json.dump(chunk_list, f, ensure_ascii=False, indent=4)

print("✅ 'parsed_result_data.json' 파일이 생성되었습니다.")

✅ 'parsed_result_readable.md' 파일이 생성되었습니다. 메모장으로 열어보세요!
✅ 'parsed_result_data.json' 파일이 생성되었습니다.


In [10]:
## 약관 검색해보기 

# 고객의 질문 (자연어)
query = "음주운전 중에 사고가 났는데, 내가 부담해야 하는 면책금이 얼마야?"

print(f"\n고객 질문: '{query}'")
print("가장 유사한 약관 조각 2개를 찾아옵니다...\n")

# DB에서 질문과 가장 의미가 비슷한 조각 2개(k=2) 검색
retrieved_docs = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(retrieved_docs):
    print(f"--- [검색된 조각 {i+1}] ---")
    print(f"📍 출처(메타데이터): {doc.metadata}")
    print(f"📄 내용:\n{doc.page_content}\n")


고객 질문: '음주운전 중에 사고가 났는데, 내가 부담해야 하는 면책금이 얼마야?'
가장 유사한 약관 조각 2개를 찾아옵니다...

--- [검색된 조각 1] ---
📍 출처(메타데이터): {'대분류': 'Q. 무면허운전이나 음주운전 시 발생한 사고도 보상받을 수 있나요?'}
📄 내용:
# Q. 무면허운전이나 음주운전 시 발생한 사고도 보상받을 수 있나요?  
A. 무면허운전이나 음주운전 사고 시 제한적으로 보상이 가능하며, 본인이 부담해야하는 금액이 높으므로 주의하시기 바랍니다.  
| 구분      | 대인 배상Ⅰ | 대인 배상Ⅱ | 대물배상 (의무) | 대물배상 (임의) | 자기신체 사고 | 무보험차 | 자기차량 손해 |
| ------- | ------ | ------ | --------- | --------- | ------- | ---- | ------- |
|         | 음주     | 사고     | 부담금       | 사고        | 부담금     | 보상   | 보상      |
| 마약 약물운전 | 사고     | 부담금    | 지급        | 부담금       | 보상      | 보상   |         |
| 무면허     | 전액     |        | 전액        |           | 면책      |      |         |
| 뺑소니     |        |        | 보상        |           |         |      |         |

--- [검색된 조각 2] ---
📍 출처(메타데이터): {'대분류': '제11조 (음주운전, 무면허운전, 마약·약물운전 또는 사고발생 시의 조치 의무 위반 관련 사고부담금)'}
📄 내용:
# 제11조 (음주운전, 무면허운전, 마약·약물운전 또는 사고발생 시의 조치 의무 위반 관련 사고부담금)  
................ 60



In [11]:
## 약관 검색해보기 

# 고객의 질문 (자연어)
query = "태풍으로 차가 침수됐는데, 자기차량손해 담보로 보상받을 수 있어?"

print(f"\n고객 질문: '{query}'")
print("가장 유사한 약관 조각 2개를 찾아옵니다...\n")

# DB에서 질문과 가장 의미가 비슷한 조각 2개(k=2) 검색
retrieved_docs = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(retrieved_docs):
    print(f"--- [검색된 조각 {i+1}] ---")
    print(f"📍 출처(메타데이터): {doc.metadata}")
    print(f"📄 내용:\n{doc.page_content}\n")


고객 질문: '태풍으로 차가 침수됐는데, 자기차량손해 담보로 보상받을 수 있어?'
가장 유사한 약관 조각 2개를 찾아옵니다...

--- [검색된 조각 1] ---
📍 출처(메타데이터): {'대분류': '[53] 자기차량손해 침수·화재 피해 한정 보상 특별약관'}
📄 내용:
# [53] 자기차량손해 침수·화재 피해 한정 보상 특별약관  
(*1) 침수: 흐르거나 고인 물, 역류하는 물, 범람하는 물, 해수 등에 피보험자동차가 빠지거나 잠기는 것을 말하며, 차량도어나 선루프 등을 개방해 놓았을 때 빗물이 들어간 것은 침수로 보지 아니합니다.  
화재: 피보험자동차가 아닌 외부에서 비롯, 시작된 화재로 피보험자 동차에 손해가 생긴 경우를 말함. 단, 피보험자동차가 다른 자동차 또는 물체와 충돌하여 발생한 화재 또는 피보험 자동차의 결함이나 기타 요인으로 피보험 자동차에서 시작된 화재로 인한 손해는 제외합니다.

--- [검색된 조각 2] ---
📍 출처(메타데이터): {'대분류': '2. 보상 내용'}
📄 내용:
# 2. 보상 내용  
보험회사는 ‘차량단독사고 손해보상 특별약관’ 제2조의 차량의 침수로 인해 보험금이 지급되는 경우(전부손해를 제외합니다.) 피보험자동차의 하체보호(언더코팅)작업을 시공한 경우에 한하여 실제 소요된 비용을 이 특별약관에서 정한 보상 한도액 내에서 지급하여 드립니다.  
| 차량 종류  | 소형승용차            | 중형승용차                  | 대형승용차      | 다목적승용차          |
| ------ | ---------------- | ---------------------- | ---------- | --------------- |
| (배기량)  | 1,600cc 이하, 전기경형 | 1,600cc 초과\~2,000cc 이하 | 2,000cc 초과 | (법정승차정원 7\~10인) |
| 보상 한도액 | 200,000원         | 250,000원               | 300,000원

In [12]:
## 약관 검색해보기 

# 고객의 질문 (자연어)
query = "사고로 상대방 차를 박았을 때, 대물배상 한도는 보통 얼마까지 설정돼?"

print(f"\n고객 질문: '{query}'")
print("가장 유사한 약관 조각 2개를 찾아옵니다...\n")

# DB에서 질문과 가장 의미가 비슷한 조각 2개(k=2) 검색
retrieved_docs = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(retrieved_docs):
    print(f"--- [검색된 조각 {i+1}] ---")
    print(f"📍 출처(메타데이터): {doc.metadata}")
    print(f"📄 내용:\n{doc.page_content}\n")


고객 질문: '사고로 상대방 차를 박았을 때, 대물배상 한도는 보통 얼마까지 설정돼?'
가장 유사한 약관 조각 2개를 찾아옵니다...

--- [검색된 조각 1] ---
📍 출처(메타데이터): {'대분류': '비고'}
📄 내용:
# 비고  
| 구 분                           | 보상 내용                   |
| ----------------------------- | ----------------------- |
| \[ 형사합의금 보상 한도액 ]             |                         |
| 교통사고처리특례법 제3조 제2항 단서에 해당하는 사고 | 보상한도                    |
| 사망의 경우                        | 2억원                     |
| 1.신호·지시위반사고                   | 상해 1\~3급의 경우 1억 5,000만원 |
| 2.중앙선침범사고                     |                         |
| 3.속도위반사고                      | 상해 4\~5급의 경우 8,000만원    |
| 4.추월방법위반사고                    | 상해 6\~7급의 경우 2,000만원    |
| 5.건널목통과방법위반사고                 | 상해 8\~11급의 경우 500만원     |
| 6.횡단보도사고                      | 상해 12\~14급의 경우 150만원    |
| 7.무면허 사고                      | 합의금                     |
| 8.주취/약물복용사고                   |                         |
| 9.인도침법사고                      |                         |
| 10.개문발차사고             